<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 3: Asymmetric Crytosystem</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Diffie-Hellman Key Exchange**

The **Diffie-Hellman Key Exchange**, published by **Whitfield Diffie and Martin Hellman** in **1976**, was the first public-key algorithm. Its only purpose is to let two users **agree on a shared secret key** over an insecure channel. Once they have the key, they can communicate with a fast **symmetric** cipher like AES.

#### **The Hard Problem: Discrete Logarithms**

Diffie-Hellman relies on the **Discrete Logarithm Problem**. If $a$ is a primitive root of a prime $p$, every integer $b$ from $1$ to $p - 1$ can be written as:

$$b \equiv a^i \pmod{p}, \qquad 0 \le i < p - 1$$

The unique exponent $i$ is the **discrete logarithm** of $b$ to the base $a$, mod $p$, written $i = \text{dlog}_{a,p}(b)$.

- **Easy direction**: given $a$, $i$ and $p$, computing $b = a^i \bmod p$ is fast
- **Hard direction**: given $b$, $a$ and $p$, finding $i$ is infeasible for a large prime $p$
- See lesson 3.3 for more detail

#### **The Key Exchange Algorithm**

Two values are agreed on and made **public**:

- $q$: a large prime number
- $\alpha$: a primitive root of $q$

```mermaid
sequenceDiagram
    participant A as User A
    participant B as User B
    Note over A,B: Public: prime q and primitive root α
    Note over A: Pick secret XA<br>YA = α^XA mod q
    Note over B: Pick secret XB<br>YB = α^XB mod q
    A->>B: YA
    B->>A: YB
    Note over A: K = YB^XA mod q
    Note over B: K = YA^XB mod q
```

| Step | User A | User B | Public or Secret? |
|------|--------|--------|-------------------|
| 1. Pick a random private value | $X_A < q$ | $X_B < q$ | **Secret** |
| 2. Compute a public value | $Y_A = \alpha^{X_A} \bmod q$ | $Y_B = \alpha^{X_B} \bmod q$ | Public |
| 3. Exchange public values | Sends $Y_A$ | Sends $Y_B$ | Public |
| 4. Compute the shared key | $K = (Y_B)^{X_A} \bmod q$ | $K = (Y_A)^{X_B} \bmod q$ | **Secret** |

#### **Why Both Users Get the Same Key**

$$
\begin{aligned}
K &= (Y_B)^{X_A} \bmod q \\
  &= (\alpha^{X_B} \bmod q)^{X_A} \bmod q \\
  &= (\alpha^{X_B})^{X_A} \bmod q \\
  &= \alpha^{X_B X_A} \bmod q \\
  &= (\alpha^{X_A})^{X_B} \bmod q \\
  &= (\alpha^{X_A} \bmod q)^{X_B} \bmod q \\
  &= (Y_A)^{X_B} \bmod q
\end{aligned}
$$

Both users end up computing $\alpha^{X_A X_B} \bmod q$. The key itself is **never sent**.

#### **Diffie-Hellman Execution for $q = 353$, $\alpha = 3$**

| Step | User A | User B |
|------|--------|--------|
| Private value | $X_A = 97$ | $X_B = 233$ |
| Public value | $Y_A = 3^{97} \bmod 353 = 40$ | $Y_B = 3^{233} \bmod 353 = 248$ |
| Shared key | $K = 248^{97} \bmod 353 = 160$ | $K = 40^{233} \bmod 353 = 160$ |

Final result:
- Both users compute $K = 160$ ✓
- An attacker only sees $q = 353$, $\alpha = 3$, $Y_A = 40$ and $Y_B = 248$
- To get the key, the attacker must find $X_A = \text{dlog}_{3,353}(40)$ or $X_B = \text{dlog}_{3,353}(248)$
- With $q = 353$ that's easy by trying every exponent, but for a 2048-bit prime it's infeasible

#### **Using Diffie-Hellman on a Network**

On a network (such as a LAN), the public values $\alpha$ and $q$ can be stored in a **central directory** that every user can access.

- **Network Setup**:
  - Each user generates their own private $X$ and stores it locally
  - Each user publishes their $Y$, or sends it when starting a conversation
  - Any two users can compute their shared key from their own $X$ and the other's $Y$

- **Static vs Ephemeral Keys**:
  - **Static**: the same $X$ is used for a long time. If it's stolen later, all past conversations can be decrypted
  - **Ephemeral**: a new $X$ is picked for every session. Stealing one doesn't expose past sessions, which is called **forward secrecy**
  - Modern protocols like TLS 1.3 use ephemeral Diffie-Hellman

#### **Man-in-the-Middle Attack**

Diffie-Hellman on its own **doesn't check who you're talking to**. An attacker who can intercept messages can swap in their own public values:

```mermaid
sequenceDiagram
    participant A as User A
    participant D as Attacker
    participant B as User B
    A->>D: YA (meant for B)
    D->>B: Attacker's own YD2
    B->>D: YB (meant for A)
    D->>A: Attacker's own YD1
    Note over A,D: A and the attacker share key K1
    Note over D,B: The attacker and B share key K2
    Note over D: Decrypts with K1, reads or changes,<br>re-encrypts with K2
```

- **Man-in-the-Middle**:
  - A thinks they share a key with B, but actually share it with the attacker, and the same for B
  - The attacker can read and change every message without either user noticing
  - **Fix**: authenticate the public values, e.g. with **digital signatures** or **certificates**

<br>

***

### **ElGamal Cryptosystem**

The **ElGamal Cryptosystem**, proposed by **Taher ElGamal** in 1985, is also based on the **Discrete Logarithm Problem**. It works very much like Diffie-Hellman, but Diffie-Hellman only **agrees on a key**, while ElGamal can **encrypt and decrypt messages** directly.

```mermaid
sequenceDiagram
    participant B as Bob (sender)
    participant A as Alice (receiver)
    Note over A: Key generation<br>Private X, public Y = α^X mod q
    A->>B: Public key {q, α, Y}
    Note over B: Pick random k<br>K = Y^k mod q<br>C1 = α^k mod q<br>C2 = K·M mod q
    B->>A: Ciphertext (C1, C2)
    Note over A: K = C1^X mod q<br>M = C2·K⁻¹ mod q
```

#### **Key Generation**

Like Diffie-Hellman, two values are agreed on and made **public**:

- $q$: a large prime number
- $\alpha$: a primitive root of $q$

- **Each User**:
  - Picks a random **private key** $X$, with $1 \le X < q - 1$
  - Computes the **public key** $Y = \alpha^X \bmod q$
  - Shares $\{q, \alpha, Y\}$ with anyone who wants to send them a message

#### **Encryption**

To send a message to Alice, Bob needs Alice's public key $\{q, \alpha, Y_A\}$.

- **Encryption Steps**:
  - Step 1: Represent the message as an integer $0 \le M < q$
    - Longer messages are split into blocks, each smaller than $q$
  - Step 2: Pick a random $k$ with $1 \le k < q - 1$
  - Step 3: Compute a **one-time key** $K = (Y_A)^k \bmod q$
  - Step 4: Encrypt $M$ as a **pair of integers** $(C_1, C_2)$:

$$C_1 = \alpha^k \bmod q \qquad C_2 = KM \bmod q$$

#### **Decryption**

- **Decryption Steps**:
  - Step 1: Recover the one-time key using the private key: $K = (C_1)^{X_A} \bmod q$
  - Step 2: Find $K^{-1} \bmod q$, using the Extended Euclidean Algorithm
  - Step 3: Recover the message: $M = (C_2 K^{-1}) \bmod q$

#### **Why It Works**

Alice can rebuild $K$ from $C_1$ without knowing $k$:

$$K = (Y_A)^k \bmod q = (\alpha^{X_A})^k \bmod q = (\alpha^k)^{X_A} \bmod q = (C_1)^{X_A} \bmod q$$

Once she has $K$, multiplying by $K^{-1}$ cancels it out:

$$C_2 K^{-1} \bmod q = K M K^{-1} \bmod q = M$$

#### **ElGamal Execution for $q = 19$, $\alpha = 10$**

The primitive roots of 19 are $\{2, 3, 10, 13, 14, 15\}$, so $\alpha = 10$ is valid.

| Phase | Who | Working | Result |
|-------|-----|---------|--------|
| Key generation | Alice | Private $X_A = 5$ | |
| | Alice | $Y_A = 10^5 \bmod 19$ | $Y_A = 3$ |
| | Alice | Public key $\{q, \alpha, Y_A\}$ | $\{19, 10, 3\}$ |
| Encryption ($M = 17$) | Bob | Random $k = 6$ | |
| | Bob | $K = 3^6 \bmod 19$ | $K = 7$ |
| | Bob | $C_1 = 10^6 \bmod 19$ | $C_1 = 11$ |
| | Bob | $C_2 = 7 \times 17 \bmod 19 = 119 \bmod 19$ | $C_2 = 5$ |
| Decryption | Alice | $K = 11^5 \bmod 19$ | $K = 7$ |
| | Alice | $7^{-1} \bmod 19$ (since $7 \times 11 = 77 = 4 \times 19 + 1$) | $K^{-1} = 11$ |
| | Alice | $M = 5 \times 11 \bmod 19 = 55 \bmod 19$ | $M = 17$ ✓ |

Final result:
- Ciphertext: $(C_1, C_2) = (11, 5)$
- Decrypted message: $17$

#### **Properties of ElGamal**

- **Ciphertext is twice as long**: each block $M$ becomes two numbers, $C_1$ and $C_2$
- **Randomised**: because $k$ is random, the same message encrypts to a **different ciphertext** every time
- **Never reuse $k$**: every block must use a new $k$
  - With the same $k$, every block gets the same $K$, and $C_1$ is identical, so an attacker can spot the reuse
  - If the attacker knows one plaintext block $M_1$, they get $K = C_2 M_1^{-1} \bmod q$ and can decrypt every other block

| Feature            | Diffie-Hellman              | ElGamal                          |
|--------------------|-----------------------------|----------------------------------|
| Hard problem       | Discrete logarithm          | Discrete logarithm               |
| Purpose            | Agree on a shared key       | Encrypt and decrypt messages     |
| Public values      | $q$, $\alpha$, $Y$          | $q$, $\alpha$, $Y$               |
| Per-message random value | -                     | $k$, new for every block         |
| Output             | A shared key $K$            | Ciphertext pair $(C_1, C_2)$     |



In [1]:
# ElGamal encryption in Python, rewritten from the GeeksforGeeks version:
# https://www.geeksforgeeks.org/elgamal-encryption-algorithm/
# For learning only. Real ElGamal uses primes of 2048 bits or more.
import secrets


def prime_factors(n):
    """Distinct prime factors of n by trial division (fine for small n)."""
    factors, d = set(), 2
    while d * d <= n:
        while n % d == 0:
            factors.add(d)
            n //= d
        d += 1
    if n > 1:
        factors.add(n)
    return factors


def is_primitive_root(g, q):
    """g is a primitive root of prime q if g^((q-1)/f) != 1 for every prime factor f of q - 1."""
    return all(pow(g, (q - 1) // f, q) != 1 for f in prime_factors(q - 1))


# ---------- the three phases ----------
def generate_keys(q, alpha):
    X = secrets.randbelow(q - 2) + 1        # private key: 1 <= X < q - 1
    Y = pow(alpha, X, q)                    # public key:  Y = alpha^X mod q
    return X, Y


def encrypt(M, q, alpha, Y, k=None):
    """Encrypt one block 0 <= M < q. A new random k is used for every block."""
    if k is None:
        k = secrets.randbelow(q - 2) + 1
    K = pow(Y, k, q)                        # one-time key K = Y^k mod q
    C1 = pow(alpha, k, q)                   # C1 = alpha^k mod q
    C2 = (K * M) % q                        # C2 = K * M mod q
    return C1, C2


def decrypt(C1, C2, q, X):
    K = pow(C1, X, q)                       # recover K = C1^X mod q
    return (C2 * pow(K, -1, q)) % q         # M = C2 * K^-1 mod q


# ---------- Example 1: the numbers from the notes ----------
q, alpha = 19, 10
X, Y = 5, pow(10, 5, 19)
C1, C2 = encrypt(17, q, alpha, Y, k=6)
print(f"q = {q}, alpha = {alpha}, primitive root: {is_primitive_root(alpha, q)}")
print(f"Alice: private X = {X}, public Y = {Y}")
print(f"Bob encrypts M = 17 with k = 6 -> (C1, C2) = ({C1}, {C2})")
print(f"Alice decrypts -> M = {decrypt(C1, C2, q, X)}")

# ---------- Example 2: a text message, one block per character ----------
q = 1000003                                  # prime larger than any byte value
alpha = next(g for g in range(2, q) if is_primitive_root(g, q))
X, Y = generate_keys(q, alpha)
message = "encryption"
ciphertext = [encrypt(ord(c), q, alpha, Y) for c in message]
decrypted = "".join(chr(decrypt(C1, C2, q, X)) for C1, C2 in ciphertext)
print(f"\nq = {q}, alpha = {alpha}")
print("Ciphertext:", ciphertext[:3], "...")
print("Decrypted: ", decrypted)

# ---------- Why k must never be reused ----------
q, alpha, X, Y = 19, 10, 5, 3
M1, M2 = 17, 4
c_a = encrypt(M1, q, alpha, Y, k=6)
c_b = encrypt(M2, q, alpha, Y, k=6)          # same k again: a mistake
K = (c_a[1] * pow(M1, -1, q)) % q            # attacker knows M1, so recovers K
print(f"\nSame k reused: C1 values {c_a[0]} and {c_b[0]} match, so the attacker spots it")
print(f"Attacker knows M1 = {M1}, recovers K = {K}, then M2 = {(c_b[1] * pow(K, -1, q)) % q}")

q = 19, alpha = 10, primitive root: True
Alice: private X = 5, public Y = 3
Bob encrypts M = 17 with k = 6 -> (C1, C2) = (11, 5)
Alice decrypts -> M = 17

q = 1000003, alpha = 2
Ciphertext: [(214205, 91909), (264168, 614507), (281822, 873301)] ...
Decrypted:  encryption

Same k reused: C1 values 11 and 11 match, so the attacker spots it
Attacker knows M1 = 17, recovers K = 7, then M2 = 4


<br>

***

### **RSA Cryptosystem**

**RSA**, named after **Rivest, Shamir and Adleman**, was invented in **1977**. It is one of the most widely used public-key cryptosystems. RSA works on **blocks of plaintext and ciphertext**, each represented as an **integer between $0$ and $n - 1$**. Its security relies on the **Integer Factorisation Problem**, and it uses **modular exponentiation** to encrypt and decrypt.

- **Key Sizes**:
  - $n$ is usually **2048 bits or more** (about 617 decimal digits)
  - 1024-bit keys (about 309 digits) are no longer considered safe
  - These keys are much larger than symmetric keys, but RSA's simplicity and its strong hard problem made it popular

```mermaid
flowchart TB
    RNG["Random number generator"] --> S1["① Choose e, p, q"]
    S1 -- "n = pq" --> S2["② Public key {e, n}"]
    S1 -- "φ(n) = (p − 1)(q − 1)<br>d = e⁻¹ mod φ(n)" --> S6["⑥ Private key {d, n}"]
    S3["③ Plaintext → decimal string"] --> S4["④ Blocks P₁, P₂, ..."]
    S4 --> S5["⑤ Ciphertext<br>Cᵢ = Pᵢᵉ mod n"]
    S2 --> S5
    S5 -- "Transmit" --> S7["⑦ Recovered text<br>Pᵢ = Cᵢᵈ mod n"]
    S6 --> S7
```

#### **Key Generation**

- **Steps**:
  - Step 1: Pick two large primes $p$ and $q$, of similar bit length, with $p \neq q$
  - Step 2: Compute $n = p \cdot q$
  - Step 3: Compute Euler's totient $\phi(n) = (p - 1)(q - 1)$
  - Step 4: Choose $e$ with $1 < e < \phi(n)$ and $\gcd(\phi(n), e) = 1$
    - In practice, $e = 65537$ is almost always used
  - Step 5: Compute $d \equiv e^{-1} \pmod{\phi(n)}$, using the Extended Euclidean Algorithm
  - Step 6: Publish the **public key** $PU = \{e, n\}$ and keep the **private key** $PR = \{d, n\}$ secret
  - $p$, $q$ and $\phi(n)$ must also stay secret, since anyone who knows them can compute $d$

#### **Encryption and Decryption**

To send Bob a message, Alice uses **Bob's public key**. The message is split into blocks so that each block $M < n$.

$$\text{Encryption:} \quad C = M^e \bmod n$$

$$\text{Decryption:} \quad M = C^d \bmod n$$

#### **Why It Works**

Since $d$ is the inverse of $e$ mod $\phi(n)$, we have $ed = 1 + k\phi(n)$ for some integer $k$. Then:

$$C^d = (M^e)^d = M^{ed} = M^{1 + k\phi(n)} = M \cdot (M^{\phi(n)})^k \equiv M \cdot 1^k = M \pmod{n}$$

The step $M^{\phi(n)} \equiv 1 \pmod{n}$ is **Euler's theorem**, from lesson 2.6.

#### **Why It's Secure**

| An Attacker Knows | To Get $d$, They Need | Which Requires          | Feasible?             |
|-------------------|-----------------------|-------------------------|-----------------------|
| $e$, $n$          | $\phi(n)$             | Factoring $n$ into $p$, $q$ | No, for large $n$ |

#### **RSA Execution: Key Generation**

| Step | Working | Result |
|------|---------|--------|
| Choose primes | $p = 73$, $q = 151$ | |
| Compute $n$ | $73 \times 151$ | $n = 11023$ |
| Compute $\phi(n)$ | $72 \times 150$ | $\phi(n) = 10800$ |
| Choose $e$ | $\gcd(11, 10800) = 1$ | $e = 11$ |
| Compute $d$ | $11^{-1} \bmod 10800$ | $d = 5891$ |
| Check | $11 \times 5891 = 64801 = 6 \times 10800 + 1$ | ✓ |
| Keys | | $PU = \{11, 11023\}$, $PR = \{5891, 11023\}$ |

#### **RSA Execution: Encrypting "how are you"**

Each character becomes a two-digit number: $a = 00, b = 01, \ldots, z = 25$, space $= 26$. Two characters make one 4-digit block, and the largest possible block, 2626, is smaller than $n = 11023$. The 11 characters are padded with a space to make 6 blocks.

| Characters | Block $M$ | $C = M^{11} \bmod 11023$ | $M = C^{5891} \bmod 11023$ |
|------------|-----------|--------------------------|----------------------------|
| h o        | 0714      | 4051                     | 0714                       |
| w ␣        | 2226      | 6953                     | 2226                       |
| a r        | 0017      | 1782                     | 0017                       |
| e ␣        | 0426      | 5202                     | 0426                       |
| y o        | 2414      | 10032                    | 2414                       |
| u ␣        | 2026      | 5799                     | 2026                       |

Final result:
- Ciphertext: `4051 6953 1782 5202 10032 5799`
- Decrypted: `how are you` ✓

#### **RSA in Practice**

- **Textbook RSA is deterministic**: the same block always gives the same ciphertext
  - An attacker can guess a likely message, encrypt it with the public key, and compare
  - Real RSA adds **random padding** first (OAEP), so each encryption is different
- **Small $n$ is unsafe**: with tiny primes, some messages don't change at all when encrypted
- **Slow**: RSA is mainly used to encrypt **symmetric keys** and to create **digital signatures**, not bulk data
- **Faster decryption**: real implementations use the **Chinese Remainder Theorem** (lesson 2.8), working mod $p$ and mod $q$ separately



In [2]:
# RSA in Python, rewritten from the Java version on GeeksforGeeks:
# https://www.geeksforgeeks.org/java-program-to-implement-the-rsa-algorithm/
# "Textbook" RSA for learning only. Real RSA adds random padding (OAEP).
import secrets
from math import gcd


# ---------- key generation ----------
def generate_keys(p, q, e=None):
    n = p * q
    phi = (p - 1) * (q - 1)
    if e is None:
        e = 65537 if gcd(65537, phi) == 1 else next(x for x in range(3, phi, 2) if gcd(x, phi) == 1)
    assert gcd(e, phi) == 1, "e must be relatively prime to phi(n)"
    d = pow(e, -1, phi)                  # d = e^-1 mod phi(n), via the Extended Euclidean Algorithm
    return (e, n), (d, n)                # public key PU, private key PR


def encrypt(M, public_key):
    e, n = public_key
    assert 0 <= M < n, "each block must be smaller than n"
    return pow(M, e, n)                  # C = M^e mod n


def decrypt(C, private_key):
    d, n = private_key
    return pow(C, d, n)                  # M = C^d mod n


# ---------- text <-> blocks: a=00, b=01, ..., z=25, space=26, 2 letters per block ----------
ALPHABET = "abcdefghijklmnopqrstuvwxyz "


def text_to_blocks(text):
    digits = "".join(f"{ALPHABET.index(c):02d}" for c in text.lower())
    if len(digits) % 4:
        digits += "26"                   # pad with a space
    return [int(digits[i:i + 4]) for i in range(0, len(digits), 4)]


def blocks_to_text(blocks):
    digits = "".join(f"{b:04d}" for b in blocks)
    return "".join(ALPHABET[int(digits[i:i + 2])] for i in range(0, len(digits), 2))


# ---------- Miller-Rabin (lesson 3.2) to generate large primes ----------
def is_probable_prime(n, rounds=40):
    if n < 4:
        return n in (2, 3)
    if n % 2 == 0:
        return False
    k, q = 0, n - 1
    while q % 2 == 0:
        k, q = k + 1, q // 2
    for _ in range(rounds):
        a = secrets.randbelow(n - 3) + 2
        x = pow(a, q, n)
        if x in (1, n - 1):
            continue
        for _ in range(k - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False                 # definitely composite
    return True                          # probably prime


def random_prime(bits):
    while True:
        candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 1   # right size, odd
        if is_probable_prime(candidate):
            return candidate


# ---------- Example 1: the key from the notes ----------
PU, PR = generate_keys(73, 151, e=11)
print("Public key PU = {e, n} =", PU)
print("Private key PR = {d, n} =", PR)

message = "how are you"
blocks = text_to_blocks(message)
cipher = [encrypt(M, PU) for M in blocks]
plain = [decrypt(C, PR) for C in cipher]
print("\nPlaintext:       ", message)
print("Plaintext blocks:", blocks)
print("Ciphertext:      ", cipher)
print("Decrypted blocks:", plain)
print("Decrypted text:  ", blocks_to_text(plain))

# ---------- Example 2: the GeeksforGeeks numbers ----------
PU, PR = generate_keys(3, 11, e=3)
print(f"\nGeeksforGeeks key: PU = {PU}, PR = {PR}")
print("Encrypt 12 ->", encrypt(12, PU), " (unchanged! n is far too small)")

# ---------- Example 3: a realistic 2048-bit key ----------
p, q = random_prime(1024), random_prime(1024)
while p == q:
    q = random_prime(1024)
PU, PR = generate_keys(p, q)
M = int.from_bytes("how are you".encode(), "big")
C = encrypt(M, PU)
print(f"\n2048-bit key: n has {len(str(PU[1]))} digits, e = {PU[0]}")
print("Decrypted:", decrypt(C, PR).to_bytes(11, "big").decode())

Public key PU = {e, n} = (11, 11023)
Private key PR = {d, n} = (5891, 11023)

Plaintext:        how are you
Plaintext blocks: [714, 2226, 17, 426, 2414, 2026]
Ciphertext:       [4051, 6953, 1782, 5202, 10032, 5799]
Decrypted blocks: [714, 2226, 17, 426, 2414, 2026]
Decrypted text:   how are you 

GeeksforGeeks key: PU = (3, 33), PR = (7, 33)
Encrypt 12 -> 12  (unchanged! n is far too small)

2048-bit key: n has 617 digits, e = 65537
Decrypted: how are you


<br>

***

### **Elliptic Curve Cryptography (ECC)**

RSA is still widely used, but as computers get faster, **RSA keys must keep getting longer** to stay secure. This adds overhead for websites and other applications that handle many secure connections.

**Elliptic Curve Cryptography (ECC)** gives the **same security as RSA with much smaller keys**. It was slower to be adopted, but it's now mainstream, especially on phones, smart cards and other small devices.

| Security Level | RSA Key Size | ECC Key Size |
|----------------|--------------|--------------|
| 112 bits       | 2048 bits    | 224 bits     |
| 128 bits       | 3072 bits    | 256 bits     |
| 192 bits       | 7680 bits    | 384 bits     |
| 256 bits       | 15360 bits   | 512 bits     |

ECC is based on a version of the **discrete logarithm problem** that uses **points on an elliptic curve** instead of numbers. For cryptography, the curve's variables and coefficients come from a **finite field**. We first look at curves over the **real numbers**, which are easy to picture, then over two finite fields: $Z_p$ and $GF(2^m)$.

#### **Elliptic Curves over Real Numbers**

Despite the name, elliptic curves are **not ellipses**. The general form is the **Weierstrass equation**:

$$y^2 + axy + by = x^3 + cx^2 + dx + e$$

For cryptography, it's simplified to:

$$y^2 = x^3 + ax + b$$

This is a **cubic** equation, since the highest power is 3. Solving for $y$ gives $y = \pm\sqrt{x^3 + ax + b}$, so every point $(x, y)$ has a mirror point $(x, -y)$. This makes the curve **symmetric about the x-axis**.

![image.png](attachment:image.png)

The set $E(a, b)$ contains all points on the curve, plus a special **Point at Infinity** $O$, also called the zero point. The curve must satisfy:

$$4a^3 + 27b^2 \neq 0$$

This rules out curves with sharp points or self-crossings.

#### **Rules for Adding Points**

Points on the curve can be **added** to get another point on the curve:

- **Identity**: $O$ acts like zero, so $P + O = P$
- **Negative**: if $P = (x, y)$, then $-P = (x, -y)$, and $P + (-P) = O$
- **Three points on a line**: if three points lie on a straight line, their sum is $O$
- **Adding $P + Q$** (different x-coordinates):
  - Draw a line through $P$ and $Q$, which hits the curve at exactly one more point $R$
  - Reflect $R$ in the x-axis: $P + Q = -R$
- **Doubling $P + P = 2P$**:
  - Draw the **tangent** at $P$, which hits the curve at one more point $S$
  - Reflect $S$ in the x-axis: $2P = -S$
- **Multiplication** is repeated addition: $4P = P + P + P + P$

#### **Addition Formulas**

For $P = (x_P, y_P)$ and $Q = (x_Q, y_Q)$ with $P \neq -Q$, the sum $R = P + Q$ is:

$$x_R = \lambda^2 - x_P - x_Q \qquad y_R = \lambda(x_P - x_R) - y_P$$

where $\lambda$ is the **slope** of the line:

$$\lambda = \begin{cases} \dfrac{y_Q - y_P}{x_Q - x_P} & \text{if } P \neq Q \quad \text{(line through } P \text{ and } Q\text{)} \\[2ex] \dfrac{3x_P^2 + a}{2y_P} & \text{if } P = Q \quad \text{(tangent at } P\text{, needs } y_P \neq 0\text{)} \end{cases}$$

#### **Elliptic Curves over Finite Fields**

For cryptography, curves are defined over **finite fields**. These can't be drawn as smooth curves, only as scattered points, so they're worked with algebraically.

| Type            | Field       | Values                        | Best Suited To |
|-----------------|-------------|-------------------------------|----------------|
| Prime curves    | $Z_p$       | Integers $0$ to $p - 1$, arithmetic mod $p$ | Software |
| Binary curves   | $GF(2^m)$   | $m$-bit binary polynomials   | Hardware (uses fast bit operations) |

#### **Elliptic Curves over $Z_p$**

$$y^2 \bmod p = (x^3 + ax + b) \bmod p$$

The same addition formulas work, with every step done **mod $p$**. Division means multiplying by the **modular inverse**:

$$\lambda = \begin{cases} (y_Q - y_P)(x_Q - x_P)^{-1} \bmod p & \text{if } P \neq Q \\ (3x_P^2 + a)(2y_P)^{-1} \bmod p & \text{if } P = Q \end{cases}$$

$$x_R = (\lambda^2 - x_P - x_Q) \bmod p \qquad y_R = (\lambda(x_P - x_R) - y_P) \bmod p$$

The curve is valid if $(4a^3 + 27b^2) \bmod p \neq 0$, and $-P = (x_P, p - y_P)$.

#### **Points on $E_{23}(1, 1)$**

With $p = 23$ and $a = b = 1$, the curve $y^2 \equiv x^3 + x + 1 \pmod{23}$ has **27 points plus $O$**, so 28 points in total.

| | | |
|---|---|---|
| (0, 1) | (6, 4) | (12, 19) |
| (0, 22) | (6, 19) | (13, 7) |
| (1, 7) | (7, 11) | (13, 16) |
| (1, 16) | (7, 12) | (17, 3) |
| (3, 10) | (9, 7) | (17, 20) |
| (3, 13) | (9, 16) | (18, 3) |
| (4, 0) | (11, 3) | (18, 20) |
| (5, 4) | (11, 20) | (19, 5) |
| (5, 19) | (12, 4) | (19, 18) |

![image-2.png](attachment:image-2.png)

The points come in pairs $(x, y)$ and $(x, 23 - y)$, so they're symmetric about $y = 11.5$. The only exception is $(4, 0)$, which is its own negative.

#### **Point Addition Execution on $E_{23}(1, 1)$**

| Operation | Slope $\lambda$ | $x_R$ | $y_R$ | Result |
|-----------|-----------------|-------|-------|--------|
| $P + Q = (3, 10) + (9, 7)$ | $\frac{7 - 10}{9 - 3} = -3 \cdot 6^{-1} = -3 \cdot 4 = -12 \equiv 11$ | $11^2 - 3 - 9 = 109 \equiv 17$ | $11(3 - 17) - 10 = -164 \equiv 20$ | $(17, 20)$ |
| $2P = 2 \cdot (3, 10)$ | $\frac{3 \cdot 3^2 + 1}{2 \cdot 10} = 28 \cdot 20^{-1} \equiv 5 \cdot 15 = 75 \equiv 6$ | $6^2 - 3 - 3 = 30 \equiv 7$ | $6(3 - 7) - 10 = -34 \equiv 12$ | $(7, 12)$ |

#### **Number of Points**

For security, the curve needs **lots of points**. The number of points $N$ on $E_p(a, b)$ is bounded by **Hasse's theorem**:

$$p + 1 - 2\sqrt{p} \le N \le p + 1 + 2\sqrt{p}$$

So $N$ is roughly $p$. For $p = 23$: $14.4 \le N \le 33.6$, and $N = 28$ ✓.

#### **Elliptic Curves over $GF(2^m)$**

$GF(2^m)$ has $2^m$ elements. Binary curves use a different equation, where $x$ and $y$ are variables and $a$, $b$ are elements of $GF(2^m)$, with $b \neq 0$:

$$y^2 + xy = x^3 + ax^2 + b$$

In $GF(2^m)$, addition and subtraction are both XOR, so $+$ and $-$ are the same. The rules are:

- $P + O = P$
- $-P = (x_P, x_P + y_P)$, and $P + (-P) = O$
- **Adding** $P + Q$ ($P \neq \pm Q$):

$$\lambda = \frac{y_Q + y_P}{x_Q + x_P} \qquad x_R = \lambda^2 + \lambda + x_P + x_Q + a \qquad y_R = \lambda(x_P + x_R) + x_R + y_P$$

- **Doubling** $R = 2P$:

$$\lambda = x_P + \frac{y_P}{x_P} \qquad x_R = \lambda^2 + \lambda + a \qquad y_R = x_P^2 + (\lambda + 1)x_R$$

#### **The Hard Problem**

Multiplying a point by a number, $Q = kP$, is **fast** using double-and-add, the curve version of square-and-multiply. Going backwards, finding $k$ from $P$ and $Q$, is the **Elliptic Curve Discrete Logarithm Problem**, and it's infeasible for large curves.

- **ECC vs Ordinary Discrete Logs**:
  - Ordinary discrete logs have faster attacks that take advantage of how integers work
  - Those attacks don't work on elliptic curve points, so ECC needs much **smaller keys** for the same security

#### **Key Exchange (ECDH)**

ECC key exchange works the same way as **Diffie-Hellman**, but with points instead of numbers.

- **Public Setup**:
  - Choose $q$: either a prime $p$ or $2^m$
  - Choose $a$ and $b$ to define the curve $E_q(a, b)$
  - Choose a **generator point** $G$ with a large **order** $n$, the smallest positive $n$ with $nG = O$
  - In practice, everyone uses **standard curves** (e.g. P-256, Curve25519), so this is done once

```mermaid
sequenceDiagram
    participant A as User A
    participant B as User B
    Note over A,B: Public: curve Ep(a, b), generator point G, order n
    Note over A: Private nA < n<br>Public PUA = nA × G
    Note over B: Private nB < n<br>Public PUB = nB × G
    A->>B: PUA
    B->>A: PUB
    Note over A: K = nA × PUB
    Note over B: K = nB × PUA
```

| Step | User A | User B |
|------|--------|--------|
| Private key (a number) | $n_A < n$ | $n_B < n$ |
| Public key (a point) | $PU_A = n_A \cdot G$ | $PU_B = n_B \cdot G$ |
| Shared key | $K = n_A \cdot PU_B$ | $K = n_B \cdot PU_A$ |

Both get the same point, because $n_A \cdot (n_B \cdot G) = n_B \cdot (n_A \cdot G)$.

#### **ECDH Execution on $E_{23}(1, 1)$ with $G = (3, 10)$**

$G$ has order $n = 28$.

| Step | User A | User B |
|------|--------|--------|
| Private key | $n_A = 9$ | $n_B = 13$ |
| Public key | $PU_A = 9G = (0, 1)$ | $PU_B = 13G = (1, 7)$ |
| Shared key | $K = 9 \cdot (1, 7) = (9, 16)$ | $K = 13 \cdot (0, 1) = (9, 16)$ |

#### **Encryption and Decryption**

The simplest method encodes the message as a **point** $P_m$ on the curve, then works like ElGamal. User A needs B's public key $PU_B$ and picks a random $k$.

$$\text{Encryption:} \quad C_m = \{kG,\ P_m + k \cdot PU_B\}$$

$$\text{Decryption:} \quad (P_m + k \cdot PU_B) - n_B(kG) = P_m + k(n_B G) - n_B(kG) = P_m$$

B multiplies the first point by the private key $n_B$ and subtracts it from the second, which cancels out the mask $k \cdot PU_B$.

#### **ECC Encryption Execution on $E_{23}(1, 1)$**

Using B's keys from above ($n_B = 13$, $PU_B = (1, 7)$):

| Step | Working | Result |
|------|---------|--------|
| Message point | | $P_m = (5, 4)$ |
| Random $k$ | | $k = 3$ |
| First point | $kG = 3 \cdot (3, 10)$ | $(19, 5)$ |
| Mask | $k \cdot PU_B = 3 \cdot (1, 7)$ | $(18, 20)$ |
| Second point | $P_m + k \cdot PU_B = (5, 4) + (18, 20)$ | $(9, 7)$ |
| Ciphertext | | $C_m = \{(19, 5), (9, 7)\}$ |
| Decrypt | $(9, 7) - 13 \cdot (19, 5) = (9, 7) - (18, 20)$ | $P_m = (5, 4)$ ✓ |

#### **Security of ECC**

- **Strengths**:
  - Much smaller keys for the same security: 256-bit ECC is about as strong as 3072-bit RSA
  - Faster, using less power and bandwidth, which suits phones and small devices
  - No efficient classical attack is known on well-chosen curves
- **Weaknesses**:
  - Must use a **well-chosen standard curve**: some curves are weak
  - The random $k$ must never be reused: Sony's PlayStation 3 signing key was leaked this way
  - Like RSA and Diffie-Hellman, ECC is broken by **Shor's algorithm** on a large quantum computer



In [3]:
# Elliptic curve cryptography over Z_p, written from scratch to show how it works.
# For learning only. Real ECC uses standard curves such as P-256 or Curve25519.
import secrets

O = None  # the point at infinity (the identity, "zero point")


class Curve:
    """y^2 = x^3 + ax + b (mod p)"""

    def __init__(self, a, b, p):
        assert (4 * a**3 + 27 * b**2) % p != 0, "4a^3 + 27b^2 must not be 0 mod p"
        self.a, self.b, self.p = a, b, p

    def on_curve(self, P):
        if P is O:
            return True
        x, y = P
        return (y * y - (x**3 + self.a * x + self.b)) % self.p == 0

    def points(self):
        """Every (x, y) with 0 <= x, y < p on the curve (only practical for small p)."""
        return [(x, y) for x in range(self.p) for y in range(self.p) if self.on_curve((x, y))]

    def neg(self, P):
        return O if P is O else (P[0], (-P[1]) % self.p)

    def add(self, P, Q):
        """Point addition and doubling using the lambda formulas."""
        if P is O:
            return Q                                      # O + Q = Q
        if Q is O:
            return P                                      # P + O = P
        if P == self.neg(Q):
            return O                                      # P + (-P) = O
        p = self.p
        (xP, yP), (xQ, yQ) = P, Q
        if P != Q:
            lam = (yQ - yP) * pow(xQ - xP, -1, p) % p     # slope of the line through P and Q
        else:
            lam = (3 * xP**2 + self.a) * pow(2 * yP, -1, p) % p   # slope of the tangent at P
        xR = (lam**2 - xP - xQ) % p
        yR = (lam * (xP - xR) - yP) % p
        return (xR, yR)

    def mul(self, k, P):
        """k * P by double-and-add (like square-and-multiply for modular exponentiation)."""
        result = O
        while k:
            if k & 1:
                result = self.add(result, P)
            P = self.add(P, P)
            k >>= 1
        return result

    def order(self, G):
        """Smallest n > 0 with nG = O (only practical for small curves)."""
        n, P = 1, G
        while P is not O:
            P, n = self.add(P, G), n + 1
        return n


# ---------- Example 1: the points of E_23(1, 1) ----------
E = Curve(a=1, b=1, p=23)
pts = E.points()
print(f"E_23(1,1) has {len(pts)} points plus O, so {len(pts) + 1} in total")
print(pts)

# ---------- Example 2: addition and doubling ----------
P, Q = (3, 10), (9, 7)
print(f"\nP + Q = {P} + {Q} = {E.add(P, Q)}")
print(f"2P = {E.add(P, P)}")
print(f"P + (-P) = {E.add(P, E.neg(P))} (the point at infinity O)")

# ---------- Example 3: ECC Diffie-Hellman on E_23(1, 1) ----------
G = (3, 10)
n = E.order(G)
nA, nB = 9, 13                                  # private keys, smaller than n
PUA, PUB = E.mul(nA, G), E.mul(nB, G)           # public keys are points
print(f"\nG = {G}, order n = {n}")
print(f"User A: private nA = {nA}, public PUA = nA*G = {PUA}")
print(f"User B: private nB = {nB}, public PUB = nB*G = {PUB}")
print(f"A computes K = nA*PUB = {E.mul(nA, PUB)}")
print(f"B computes K = nB*PUA = {E.mul(nB, PUA)}")

# ---------- Example 4: ECC encryption (ElGamal style) ----------
Pm = (5, 4)                                     # message encoded as a point on the curve
k = 3                                           # random, new for every message
C1, C2 = E.mul(k, G), E.add(Pm, E.mul(k, PUB))  # Cm = {kG, Pm + k*PUB}
decrypted = E.add(C2, E.neg(E.mul(nB, C1)))     # Pm = C2 - nB*C1
print(f"\nEncrypt Pm = {Pm} with k = {k}: Cm = {{{C1}, {C2}}}")
print(f"B decrypts: C2 - nB*C1 = {decrypted}")

# ---------- Example 5: a real-size curve (NIST P-256) ----------
P256 = Curve(
    a=-3,
    b=0x5AC635D8AA3A93E7B3EBBD55769886BC651D06B0CC53B0F63BCE3C3E27D2604B,
    p=2**256 - 2**224 + 2**192 + 2**96 - 1,
)
G256 = (0x6B17D1F2E12C4247F8BCE6E563A440F277037D812DEB33A0F4A13945D898C296,
        0x4FE342E2FE1A7F9B8EE7EB4A7C0F9E162BCE33576B315ECECBB6406837BF51F5)
n256 = 0xFFFFFFFF00000000FFFFFFFFFFFFFFFFBCE6FAADA7179E84F3B9CAC2FC632551
a_priv, b_priv = secrets.randbelow(n256 - 1) + 1, secrets.randbelow(n256 - 1) + 1
A_pub, B_pub = P256.mul(a_priv, G256), P256.mul(b_priv, G256)
print(f"\nP-256: 256-bit keys, about as strong as 3072-bit RSA")
print("Shared keys match:", P256.mul(a_priv, B_pub) == P256.mul(b_priv, A_pub))

E_23(1,1) has 27 points plus O, so 28 in total
[(0, 1), (0, 22), (1, 7), (1, 16), (3, 10), (3, 13), (4, 0), (5, 4), (5, 19), (6, 4), (6, 19), (7, 11), (7, 12), (9, 7), (9, 16), (11, 3), (11, 20), (12, 4), (12, 19), (13, 7), (13, 16), (17, 3), (17, 20), (18, 3), (18, 20), (19, 5), (19, 18)]

P + Q = (3, 10) + (9, 7) = (17, 20)
2P = (7, 12)
P + (-P) = None (the point at infinity O)

G = (3, 10), order n = 28
User A: private nA = 9, public PUA = nA*G = (0, 1)
User B: private nB = 13, public PUB = nB*G = (1, 7)
A computes K = nA*PUB = (9, 16)
B computes K = nB*PUA = (9, 16)

Encrypt Pm = (5, 4) with k = 3: Cm = {(19, 5), (9, 7)}
B decrypts: C2 - nB*C1 = (5, 4)

P-256: 256-bit keys, about as strong as 3072-bit RSA
Shared keys match: True


<br>

***

### **Probabilistic Encryption**

Most of the algorithms covered so far are **Deterministic**: the same key and the same plaintext always give the **same ciphertext**. This leaks information. An attacker can spot repeated messages, or guess a likely message, encrypt it with the public key and compare. Some schemes add randomness to avoid this (ElGamal's random $k$, CBC's random IV), but textbook RSA doesn't.

**Probabilistic Encryption** adds randomness on purpose. The **same key and the same plaintext give a different ciphertext every time**, but the receiver can still decrypt all of them.

- **Probabilistic Encryption**:
  - **Advantage**: an attacker can't tell if the same message was sent twice, or test guesses against the ciphertext
  - **Disadvantage**: the ciphertext is much larger than the plaintext (**message expansion**)

This lesson covers the **Goldwasser-Micali (GM) cryptosystem** (1982), the first probabilistic public-key scheme to be **provably secure**. Its security rests on the **Quadratic Residuosity Problem** from lesson 3.4.

#### **Recap: Quadratic Residues**

- **Quadratic Residue mod $p$**: $a$ is a residue if $x^2 \equiv a \pmod{p}$ has a solution
  - For an odd prime $p$, exactly half the numbers from $1$ to $p - 1$ are residues: $|Q_R| = |Q_{NR}| = \frac{p - 1}{2}$

- **Legendre Symbol** (for a prime $p$, with $\gcd(a, p) = 1$):

$$\left(\frac{a}{p}\right) = \begin{cases} 1 & \text{if } a \in Q_R \\ -1 & \text{if } a \in Q_{NR} \end{cases}$$

  - Easy to compute with **Euler's criterion**: $\left(\frac{a}{p}\right) \equiv a^{(p-1)/2} \pmod{p}$

- **Jacobi Symbol** (for $n = pq$):

$$\left(\frac{a}{n}\right) = \left(\frac{a}{p}\right)\left(\frac{a}{q}\right)$$

  - Can be computed **without knowing $p$ and $q$**, using quadratic reciprocity
  - If it's $1$, $a$ is either a real residue mod $n$ or a **pseudosquare**, and only someone who knows $p$ and $q$ can tell which

- **Pseudosquare**: a non-residue mod $p$ **and** mod $q$, so its Jacobi symbol is $(-1)(-1) = 1$, even though it isn't a square

#### **How Goldwasser-Micali Works**

GM encrypts **one bit at a time**:

- A **0** bit becomes a **real square** $x^2 \bmod n$
- A **1** bit becomes a **pseudosquare** $y \cdot x^2 \bmod n$

Both have Jacobi symbol 1, so they look the same to anyone without $p$ and $q$. Because $x$ is **random for every bit**, the same bit encrypts to a different number every time.

```mermaid
flowchart LR
    B{"Plaintext bit mᵢ"} -- "0" --> C0["cᵢ = x² mod n<br>(a real square)"]
    B -- "1" --> C1["cᵢ = y·x² mod n<br>(a pseudosquare)"]
    C0 --> L{"Legendre (cᵢ / p)"}
    C1 --> L
    L -- "+1" --> M0["mᵢ = 0"]
    L -- "−1" --> M1["mᵢ = 1"]
```

Instead of a trapdoor **function**, GM uses an **unapproximable trapdoor predicate**: a yes/no question ("is $c$ a residue?") that can't even be guessed better than chance without the trapdoor $p$, $q$.

#### **Key Generation**

- **Steps**:
  - Choose two large random primes $p$ and $q$ of roughly the same size
  - Compute $n = p \cdot q$
  - Choose a **pseudosquare** $y$: a non-residue mod both $p$ and $q$, so $\left(\frac{y}{n}\right) = 1$
- **Keys**:
  - **Public key**: $PU = (n, y)$
  - **Private key**: $PR = (p, q)$

#### **Encryption**

Alice gets Bob's public key $(n, y)$ and writes the message as bits $m = m_1 m_2 \ldots m_t$. For **each bit**:

- Step 1: Pick a **random** $x$ with $\gcd(x, n) = 1$
- Step 2: Compute:

$$c_i = \begin{cases} x^2 \bmod n & \text{if } m_i = 0 \\ y \cdot x^2 \bmod n & \text{if } m_i = 1 \end{cases}$$

Alice sends $c = (c_1, c_2, \ldots, c_t)$.

#### **Decryption**

For each $c_i$, Bob uses his private key $p$ to compute the Legendre symbol:

$$e_i = \left(\frac{c_i}{p}\right) \qquad m_i = \begin{cases} 0 & \text{if } e_i = 1 \\ 1 & \text{if } e_i = -1 \end{cases}$$

#### **Goldwasser-Micali Execution for $p = 7$, $q = 11$**

**Key generation:** $n = 77$.

| Prime | Residues | Non-residues |
|-------|----------|--------------|
| 7     | 1, 2, 4  | 3, 5, 6      |
| 11    | 1, 3, 4, 5, 9 | 2, 6, 7, 8, 10 |

$y = 6$ is a non-residue mod 7 **and** mod 11, so it's a pseudosquare. Public key $(77, 6)$, private key $(7, 11)$.

**Encrypting $m = 1011$:**

| Bit $m_i$ | Random $x$ | Formula | $c_i$ |
|-----------|------------|---------|-------|
| 1 | 2 | $6 \cdot 2^2 \bmod 77$  | 24 |
| 0 | 3 | $3^2 \bmod 77$          | 9  |
| 1 | 4 | $6 \cdot 4^2 \bmod 77$  | 19 |
| 1 | 5 | $6 \cdot 5^2 \bmod 77$  | 73 |

**Decrypting with $p = 7$:**

| $c_i$ | $c_i \bmod 7$ | $(c_i \bmod 7)^3 \bmod 7$ | Legendre | Bit |
|-------|---------------|---------------------------|----------|-----|
| 24    | 3             | 6                         | −1       | 1   |
| 9     | 2             | 1                         | +1       | 0   |
| 19    | 5             | 6                         | −1       | 1   |
| 73    | 3             | 6                         | −1       | 1   |

Final result:
- Ciphertext: $(24, 9, 19, 73)$, which decrypts to $1011$ ✓
- Encrypting $1011$ again with $x = 8, 9, 10, 13$ gives $(76, 4, 61, 13)$: same message, **different ciphertext**

#### **Properties of Goldwasser-Micali**

- **Provably secure**: breaking it is as hard as the quadratic residuosity problem
- **Message expansion**: each plaintext **bit** becomes a number mod $n$, so with a 2048-bit $n$ the ciphertext is about **2048 times** larger
- **Bit by bit**: works like a stream cipher, with a block size of 1 bit
- **Not used in practice** because of the expansion, but it introduced the idea of **semantic security**, which modern schemes like RSA-OAEP and ElGamal are built to meet

<br>

***